<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: كورجون ديمتري، @tbb
    
## <center> مشروع تحليل البيانات الفردية


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_palette('Set3')

%matplotlib inline
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.model_selection import validation_curve, learning_curve
from sklearn.metrics import mean_squared_error


# وصف مجموعة البيانات والميزات 
### [رابط كاجل](https://www.kaggle.com/c/elo-merchant-category-recommendation)
Elo - واحدة من أكبر العلامات التجارية للدفع في البرازيل. في مجموعة البيانات يمكننا رؤية العملاء الذين يستخدمون Elo ومعاملاتهم. نحن بحاجة إلى التنبؤ بدرجة الولاء لكل بطاقة_id.
وصف الملفات هي
* Train.csv - مجموعة التدريب
* test.csv - مجموعة الاختبار
* Sample_submission.csv - نموذج ملف إرسال بالتنسيق الصحيح - يحتوي على جميع معرفات البطاقة التي من المتوقع أن تتوقعها.
*history_transactions.csv - ما يصل إلى 3 أشهر من المعاملات التاريخية لكل بطاقة_id
* ملف التجار.csv - معلومات إضافية حول جميع التجار / معرفات التجار الموجودة في مجموعة البيانات.
* new_merchant_transactions.csv - بيانات لمدة شهرين لكل بطاقة_id تحتوي على جميع عمليات الشراء التي أجراها_card_id في Merchant_ids والتي لم تتم زيارتها في البيانات التاريخية.
يحتوي الملفان *historical_transactions.csv* و*new_merchant_transactions.csv* على معلومات حول معاملات كل بطاقة. *historical_transactions.csv* يحتوي على ما يصل إلى 3 أشهر من المعاملات لكل بطاقة في أي من معرفات التجار المتوفرة. *new_merchant_transactions.csv* يحتوي على المعاملات التي تمت لدى التجار الجدد (معرّفات_التاجر التي لم يقم معرّف_البطاقة هذا بزيارتها بعد) على مدار فترة شهرين.
يحتوي *merchants.csv* على معلومات مجمعة لكل معرف تاجر ممثل في مجموعة البيانات.
#### مجموعة البيانات الرئيسية:


In [ ]:
train = pd.read_csv('../../data/ELO/train.csv', parse_dates=['first_active_month'])
test = pd.read_csv('../../data/ELO/test.csv', parse_dates=['first_active_month'])

train.head()

In [ ]:
# columns description
pd.read_excel('../../data/ELO/Data_Dictionary.xlsx', sheet_name='train', header=2)


#### المعاملات التاريخية:


In [ ]:
hist = pd.read_csv('../../data/ELO/historical_transactions.csv')
hist.head()

In [ ]:
# columns description
pd.read_excel('../../data/ELO/Data_Dictionary.xlsx', sheet_name='history', header=2)


#### المعاملات التجارية الجديدة


In [ ]:
transaction = pd.read_csv('../../data/ELO/new_merchant_transactions.csv')
transaction.head()

In [ ]:
# columns description
pd.read_excel('../../data/ELO/Data_Dictionary.xlsx', sheet_name='new_merchant_period', header=2)


# القليل من المعالجة المسبقة


In [ ]:
train.info()

نظرًا لأن الميزات قاطعة، فيمكننا تغيير النوع لتحرير بعض الذاكرة.


In [ ]:
train['feature_1'] = train['feature_1'].astype('category')
train['feature_2'] = train['feature_2'].astype('category')
train['feature_3'] = train['feature_3'].astype('category')

test['feature_1'] = test['feature_1'].astype('category')
test['feature_2'] = test['feature_2'].astype('category')
test['feature_3'] = test['feature_3'].astype('category')

In [ ]:
train.info()


# تحليل البيانات الاستكشافية وهندسة الميزات



#### التحقق من البيانات المفقودة


In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()


#### العمود المستهدف
لنبدأ التحليلات بالقيمة المستهدفة


In [ ]:
fig, ax = plt.subplots(figsize = (16, 6))
plt.suptitle('Target value distribution', fontsize=24)
sns.distplot(train['target'], bins=50, ax=ax);


يمكننا أن نرى أن بعض قيم الولاء متباعدة (أقل من -30) مقارنة بقيم أخرى.


In [ ]:
(train['target'] < -30).sum(), round((train['target'] < -30).sum() / train['target'].count(), 2)


إذن، هناك 2207 صفًا (حوالي 1% من البيانات)، لها قيم مختلفة عن الباقي. نظرًا لأن مقياس RMSE قد تلعب هذه الصفوف دورًا مهمًا. لذا احذر منهم.



#### الشهر النشط الأول
في هذا القسم، دعنا نرى ما إذا كان هناك أي تغيير في التوزيع بين مجموعات التدريب والاختبار فيما يتعلق بالشهر النشط الأول للبطاقة.


In [ ]:
fig, ax = plt.subplots(figsize = (14, 6))

first_month_count_train = train['first_active_month'].dt.date.value_counts().sort_index()
sns.barplot(first_month_count_train.index,
            first_month_count_train.values,
            alpha=0.8, ax=ax, color='#96CAC0')

first_month_count_test = test['first_active_month'].dt.date.value_counts().sort_index()
sns.barplot(first_month_count_test.index,
            first_month_count_test.values,
            alpha=0.8, ax=ax, color='#F6F6BC')

plt.xticks(rotation='vertical')
plt.xlabel('First active month', fontsize=12)
plt.ylabel('Number of cards', fontsize=12)
plt.title('First active month count')

plt.show()


يبدو أن التوزيع مشابه نوعًا ما بين مجموعة التدريب والاختبار. لذلك لا نحتاج حقًا إلى إجراء تقسيم على أساس الوقت على ما أعتقد.



#### ميزات مجهولة
في هذا القسم، دعونا نرى ما إذا كانت المتغيرات الأخرى في مجموعة بيانات القطار تتمتع بقدرة تنبؤية جيدة في العثور على درجة الولاء.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (16, 6))
plt.suptitle('Counts of categiories for features', fontsize=24)
sns.countplot(data=train, x='feature_1', ax=ax[0])
sns.countplot(data=train, x='feature_2', ax=ax[1]).set(ylabel=None)
sns.countplot(data=train, x='feature_3', ax=ax[2]).set(ylabel=None);

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
plt.suptitle('Violineplots for features and target', fontsize=24)
sns.violinplot(x='feature_1', y='target', data=train, ax=ax[0], title='feature_1', palette='Set3')
sns.violinplot(x='feature_2', y='target', data=train, ax=ax[1], title='feature_2', palette='Set3')
sns.violinplot(x='feature_3', y='target', data=train, ax=ax[2], title='feature_3', palette='Set3');


بالعين المجردة، يبدو توزيع الفئات المختلفة في الميزات الثلاثة متشابهًا نوعًا ما. قد تكون النماذج قادرة على العثور على شيء هنا.



الآن دعونا نصنع بعض الميزات بناءً على المعاملات التاريخية وندمجها مع مجموعة التدريب والاختبار.
#### عدد المعاملات التاريخية للبطاقة


In [ ]:
history_purchase_amount = hist.groupby('card_id')['purchase_amount'].size().reset_index()
history_purchase_amount.columns = ['card_id', 'history_purchase_amount']
train = pd.merge(train, history_purchase_amount, on='card_id', how='left')
test = pd.merge(test, history_purchase_amount, on='card_id', how='left')

In [ ]:
history_purchase_amount = train.groupby('history_purchase_amount')['target'].mean().sort_index()[:-50]
fig, ax = plt.subplots(figsize=(16, 6))
plt.suptitle('Loyalty score by Number of historical transactions', fontsize=24)
sns.lineplot(history_purchase_amount.index[::-1],
             history_purchase_amount.values[::-1],
             ax=ax);


الآن قم بإحصاء المعاملات التاريخية ثم قم ببعض المخططات المربعة لرؤية المخططات بشكل أفضل.


In [ ]:
bins = [0] + [2 ** p for p in range(4, 13)]
train['binned_history_purchase_amount'] = pd.cut(train['history_purchase_amount'], bins)

plt.figure(figsize=(16, 6))
sns.boxplot(x='binned_history_purchase_amount', y='target', data=train, showfliers=False)
plt.xticks(rotation='vertical')
plt.xlabel('binned_num_hist_transactions', fontsize=12)
plt.ylabel('Loyalty score', fontsize=12)
plt.title('Distribution of binned history purchase amount', fontsize=24)
plt.show()


#### قيمة المعاملات التاريخية
التحقق من قيمة المعاملات التاريخية للبطاقات والتحقق من توزيع نقاط الولاء بناء على ذلك.


In [ ]:
gdf = hist.groupby('card_id')['purchase_amount'].agg(['sum', 'mean', 'std', 'min', 'max']).reset_index()
gdf.columns = ['card_id', 
               'sum_history_purchase_amount', 
               'mean_history_purchase_amount', 
               'std_history_purchase_amount', 
               'min_history_purchase_amount', 
               'max_history_purchase_amount']
train = pd.merge(train, gdf, on='card_id', how='left')
test = pd.merge(test, gdf, on='card_id', how='left')

In [ ]:
bins = np.percentile(train['sum_history_purchase_amount'], range(0,101,10))
train['binned_sum_history_purchase_amount'] = pd.cut(train['sum_history_purchase_amount'], bins)

plt.figure(figsize=(16, 6))
sns.boxplot(x='binned_sum_history_purchase_amount', y='target', data=train, showfliers=False)
plt.xticks(rotation='vertical')
plt.xlabel('Binned sum history purchase amount', fontsize=12)
plt.ylabel('Loyalty score', fontsize=12)
plt.title('Sum of historical transaction value (binned) distribution', fontsize=24)
plt.show()

كما نرى، يبدو أن درجة الولاء تزداد مع `sum of historical transaction value`. هذا هو المتوقع. الآن يمكننا أن نفعل نفس المؤامرة مع `Mean value of historical transaction`.


In [ ]:
bins = np.percentile(train['mean_history_purchase_amount'], range(0,101,10))
train['binned_mean_history_purchase_amount'] = pd.cut(train['mean_history_purchase_amount'], bins)

plt.figure(figsize=(16, 6))
sns.boxplot(x='binned_mean_history_purchase_amount', y='target', data=train, showfliers=False)
plt.xticks(rotation='vertical')
plt.xlabel('Binned Mean Historical Purchase Amount', fontsize=12)
plt.ylabel('Loyalty score', fontsize=12)
plt.title('Mean of historical transaction value (binned) distribution', fontsize=24)
plt.show()


#### المعاملات التجارية الجديدة
في هذا القسم، دعونا نلقي نظرة على بيانات المعاملات التجارية الجديدة ونقوم ببعض التحليلات


In [ ]:
gdf = transaction.groupby('card_id')['purchase_amount'].size().reset_index()
gdf.columns = ['card_id', 'transactions_count']
train = pd.merge(train, gdf, on='card_id', how='left')
test = pd.merge(test, gdf, on='card_id', how='left')

In [ ]:
bins = [0, 10, 20, 30, 40, 50, 75, 10000]
train['binned_transactions_count'] = pd.cut(train['transactions_count'], bins)

plt.figure(figsize=(16, 6))
sns.boxplot(x='binned_transactions_count', y='target', data=train, showfliers=False)
plt.xticks(rotation='vertical')
plt.xlabel('Binned transactions count', fontsize=12)
plt.ylabel('Loyalty score', fontsize=12)
plt.title('Number of new merchants transaction (binned) distribution', fontsize=24)
plt.show()


يبدو أن نقاط الولاء تنخفض مع زيادة عدد المعاملات التجارية الجديدة باستثناء سلة المهملات الأخيرة.


In [ ]:
gdf = transaction.groupby('card_id')['purchase_amount'].agg(['sum', 'mean', 'std', 'min', 'max']).reset_index()
gdf.columns = ['card_id', 
               'sum_transactions_count', 
               'mean_transactions_count', 
               'std_transactions_count', 
               'min_transactions_count', 
               'max_transactions_count']
train = pd.merge(train, gdf, on='card_id', how='left')
test = pd.merge(test, gdf, on='card_id', how='left')

In [ ]:
bins = np.nanpercentile(train['sum_transactions_count'], range(0,101,10))
train['binned_sum_transactions_count'] = pd.cut(train['sum_transactions_count'], bins)

plt.figure(figsize=(16, 6))
sns.boxplot(x='binned_sum_transactions_count', y='target', data=train, showfliers=False)
plt.xticks(rotation='vertical')
plt.xlabel('binned sum of new merchant transactions', fontsize=12)
plt.ylabel('Loyalty score', fontsize=12)
plt.title('Sum of new merchants transaction value (binned) distribution', fontsize=24)
plt.show()


يبدو أن نقاط الولاء تزداد مع زيادة مجموع قيم معاملات التاجر الجديدة ولكن في سلة المهملات الأخيرة.


In [ ]:
bins = np.nanpercentile(train['mean_transactions_count'], range(0,101,10))
train['binned_mean_transactions_count'] = pd.cut(train['mean_transactions_count'], bins)

plt.figure(figsize=(16, 6))
sns.boxplot(x='binned_mean_transactions_count', y='target', data=train, showfliers=False)
plt.xticks(rotation='vertical')
plt.xlabel('binned mean of new merchant transactions', fontsize=12)
plt.ylabel('Loyalty score', fontsize=12)
plt.title('Mean of New merchants transaction value (binned) distribution', fontsize=24)
plt.show()


# الأنماط والرؤى وخصائص البيانات



لذلك، وبناء على نتائج تحليل البيانات، يمكن استخلاص الاستنتاجات التالية:
* لا توجد فجوات في بيانات القطار/الطائرات، ولكن المعلومات التفصيلية مقدمة فقط للأشهر الثلاثة الماضية، لذلك لدينا بعض البيانات المفقودة في الميزات التي تم إنشاؤها.
* هناك قيم متطرفة في المتغير المستهدف تتطلب تحليلًا إضافيًا. قد يكون هذا بمثابة حظر للاحتيال، أو، على سبيل المثال، سد الثغرات بشكل سيئ.
* انطلاقًا من اعتماد الولاء على عدد المشتريات، فإن الولاء ينمو بعدد كبير بما فيه الكفاية من المشتريات (> 75)، وقبل ذلك ينخفض ​​عادةً. وهذا أمر متوقع، لأن أولئك الذين توقفوا عند عدد قليل من المشتريات، كقاعدة عامة، غير راضين عن الخدمة.



# المعالجة المسبقة للبيانات



لقد تم إغفال صف واحد في بيانات الاختبار `first_active_month`، لذلك دعونا نصلحه.


In [ ]:
test.loc[test['first_active_month'].isna(), 'first_active_month'] = test.loc[
    (test['feature_1'] == 5) & (test['feature_2'] == 2) & (test['feature_3'] == 1),
    'first_active_month'].min()


املأ البيانات على `card_id` التي ليس لديها معاملات خلال الأشهر الثلاثة الماضية.


In [ ]:
cols_to_fill = [
    'transactions_count', 'sum_transactions_count', 
    'mean_transactions_count', 'std_transactions_count',
    'min_transactions_count', 'max_transactions_count',    
]

train[cols_to_fill] = train[cols_to_fill].fillna(0)
test[cols_to_fill] = test[cols_to_fill].fillna(0)


#إضافة عدة مميزات أخرى



هنا نضيف ميزات التاريخ المشتركة.


In [ ]:
max_date = train['first_active_month'].dt.date.max()
def process_main(df):
    date_parts = ['year', 'weekday', 'month']
    for part in date_parts:
        part_col = 'first_' + part
        df[part_col] = getattr(df['first_active_month'].dt, part).astype(int)
            
    df['elapsed_time'] = (max_date - df['first_active_month'].dt.date).dt.days
    
    return df

In [ ]:
train = process_main(train)
test = process_main(test)


# التحقق من الصحة وضبط المعلمة الفائقة


#### النموذج الأساسي
دعونا نبني نموذجًا أساسيًا باستخدام الميزات التي تم إنشاؤها حتى الآن. بادئ ذي بدء، يتعين علينا تقسيم البيانات إلى مجموعات التدريب والتحقق من الصحة.


In [ ]:
cols_to_use = [
    'feature_1', 'feature_2', 'feature_3',
    'first_year', 'first_month', 'first_weekday', 'elapsed_time',
    'history_purchase_amount', 'sum_history_purchase_amount',
    'mean_history_purchase_amount', 'std_history_purchase_amount', 
    'min_history_purchase_amount', 'max_history_purchase_amount',
    'transactions_count', 'sum_transactions_count', 
    'mean_transactions_count', 'std_transactions_count',
    'min_transactions_count', 'max_transactions_count',
]

X_train, X_holdout, y_train, y_holdout = train_test_split(train[cols_to_use],
                                                          train['target'],
                                                          test_size=0.2)
X_test = test[cols_to_use]


الآن بعد أن قمنا بإعداد البيانات، يمكننا حذف البيانات الأولية.


In [ ]:
del train, test, hist, transaction

In [ ]:
params = {
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 1.0,
    'max_depth': 3,
    'max_features': 'sqrt',
    'n_iter_no_change': 5,
    'validation_fraction': 0.2,
    'tol': 0.00001,
    'random_state': 11,
}


تناسب النموذج الأساسي


In [ ]:
%%time
model = GradientBoostingRegressor(**params)
model.fit(X_train[cols_to_use], y_train)

In [ ]:
score = mean_squared_error(y_holdout, model.predict(X_holdout))
print(f'Baseline model score: {np.sqrt(score)}')

In [ ]:
fi = list(zip(cols_to_use, model.feature_importances_))
fi = pd.DataFrame(sorted(fi, key=lambda x: x[1], reverse=True), columns=['Feature', 'Importance'])

In [ ]:
plt.figure(figsize=(16, 6))
sns.barplot(x='Importance', y='Feature', data=fi, orient='h')
plt.title('Features importance', fontsize=24);


# منحنيات التحقق والتعلم



قم بتغيير المعلمات وضبط `n_estimators` باستخدام منحنى التحقق من الصحة.


In [ ]:
params = {
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'max_depth': 7,
    'max_features': 'sqrt',
    'n_iter_no_change': 5,
    'validation_fraction': 0.2,
    'tol': 0.00001,
    'random_state': 11,
}

In [ ]:
def plot_validation_curve(model, X_train, y_train,
                          param, param_range, cv=3,
                          scoring='neg_mean_squared_error'):
    train_scores, test_scores = validation_curve(
        model, X_train, y_train, cv=cv,
        param_name=param, param_range=param_range,
        scoring=scoring, n_jobs=-1
    )
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.figure(figsize=(16, 6))
    plt.title('Validation Curve')
    plt.xlabel('n_estimators')
    plt.ylabel('Score')

    plt.semilogx(param_range, train_scores_mean, label='Training score',
                 color='darkorange', lw=2)
    plt.fill_between(param_range, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.2,
                     color='darkorange', lw=2)
    plt.semilogx(param_range, test_scores_mean, label='Cross-validation score',
                 color='navy', lw=2)
    plt.fill_between(param_range, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.2,
                     color='navy', lw=2)
    plt.legend(loc='best')
    plt.show()

In [ ]:
%%time
plot_validation_curve(GradientBoostingRegressor(**params),
                      X_train[cols_to_use], y_train,
                      param='n_estimators',
                      param_range=[10 ** x for x in range(1, 6)])


يطرح منحنى التحقق احتمالين: أولاً، أننا لا نملك نطاق المعلمة الصحيح للعثور على أفضل `n_estimators` ونحتاج إلى توسيع بحثنا ليشمل قيمًا أكبر. والثاني هو أن المعلمات الفائقة الأخرى (مثل `learning_rate` أو `max_depth`، أو حتى `subsample`) قد يكون لها تأثير أكبر على النموذج الافتراضي من `n_estimators` بحد ذاته. على الرغم من أن منحنيات التحقق من الصحة يمكن أن تعطينا بعض الحدس حول أداء النموذج لمعلمة تشعبية واحدة، إلا أن البحث في الشبكة مطلوب لفهم أداء النموذج فيما يتعلق بمعلمات تشعبية متعددة.


In [ ]:
def plot_learning_curve(model, X_train, y_train, cv=3,
                        train_sizes=None, scoring='neg_mean_squared_error',
                        random_state=11):
    if not train_sizes:
        train_sizes = np.linspace(.1, 1.0, 8)
        
    train_sizes, train_scores, test_scores = learning_curve(
        model, X_train, y_train, cv=cv,
        train_sizes=train_sizes,
        scoring=scoring,
        random_state=random_state,
        n_jobs=-1
    )

    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    
    plt.figure(figsize=(16, 6))
    plt.title('Learning curve')
    plt.xlabel('Training examples')
    plt.ylabel('Score') 
    plt.grid()
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1,
                     color='r')
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color='g')
    plt.plot(train_sizes, train_scores_mean, 'o-', color='r',
             label='Training score')
    plt.plot(train_sizes, test_scores_mean, 'o-', color='g',
             label='Cross-validation score')

    plt.legend(loc='best')
    plt.show()

In [ ]:
%%time
gbm = GradientBoostingRegressor(**params)
plot_learning_curve(gbm, X_train[cols_to_use], y_train)


يُظهر منحنى التعلم هذا تباينًا كبيرًا في الاختبار ودرجة منخفضة. يمكننا أن نرى أن درجات التدريب والاختبار لم تتقارب بعد، لذلك من المحتمل أن يستفيد هذا النموذج من المزيد من بيانات التدريب. أخيرًا، لا يعاني هذا النموذج من الخطأ بسبب التباين (درجات السيرة الذاتية لبيانات الاختبار أكثر تباينًا من بيانات التدريب) لذلك من الممكن أن يكون النموذج غير مناسب.



# التنبؤ لعينات الصمود والاختبار 


In [ ]:
%%time
new_params = params
new_params['n_iter_no_change'] = None
new_params['n_estimators'] = 100
model = GradientBoostingRegressor(**new_params)
model.fit(X_train[cols_to_use], y_train)

In [ ]:
score = mean_squared_error(y_holdout, model.predict(X_holdout))
print(f'Final model score: {np.sqrt(score)}')

In [ ]:
submission = pd.read_csv('../../data/ELO/sample_submission.csv')
submission['target'] = model.predict(X_test)
submission.to_csv('submit.csv', index=False)


# وصف المقاييس



يتم تسجيل التوقعات على أساس جذر متوسط الخطأ التربيعي. تم تعريف RMSE على النحو التالي:
$$ RMSE = \sqrt{ \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i) ^ 2 }$$
حيث $\hat{y}$ هي درجة الولاء المتوقعة لكل `card_id`، و$y$ هي درجة الولاء الفعلية المخصصة لـ `card_id`.RMSE هو الجذر التربيعي لتباين القيم المتبقية. إنه يشير إلى الملاءمة المطلقة للنموذج مع البيانات - مدى قرب نقاط البيانات المرصودة من القيم المتوقعة للنموذج. يعد RMSE مقياسًا جيدًا لمدى دقة النموذج في توقع الاستجابة، وهو المعيار الأكثر أهمية للملاءمة إذا كان الغرض الرئيسي من النموذج هو التنبؤ.



# تقييم النموذج
النتيجة - نموذج دقيق إلى حد ما (منتصف لوحة المتصدرين) مع وجود تباين بسيط.



# الاستنتاجات



خلاصة القول، لدينا النموذج بعيدًا عن التنبؤات المثالية وهناك مجال كبير للتحسين هنا. 
* أولاً، سيكون من الأفضل ضبط المعلمات بشكل أفضل (بصراحة، توقفت عن انتظار نهاية GridSearch بعد الليلة الثانية). 
* ثانيًا، إنشاء ميزات أكثر إفادة وتجربة نماذج أخرى (مثل xgboost وLightGBM وCatBoost).
إذن، لقد انتهى الوقت. شكرا لاهتمامكم!